# **Colab Pro notebook from https://github.com/markn333/fast-stable-diffusion. [ComfyUI Colab](https://colab.research.google.com/github/markn333/fast-stable-diffusion/blob/main/fast_stable_diffusion_ComfyUI.ipynb)**

In [1]:
#@markdown # Connect Google Drive
from google.colab import drive
from IPython.display import clear_output
import ipywidgets as widgets
import os

def inf(msg, style, wdth): inf = widgets.Button(description=msg, disabled=True, button_style=style, layout=widgets.Layout(min_width=wdth));display(inf)
Shared_Drive = "" #@param {type:"string"}
#@markdown - Leave empty if you're not using a shared drive

print("[0;33mConnecting...")
def drive_alive():
  # a dead FUSE mount still looks mounted; only a real access reveals it
  try:
    os.listdir('/content/gdrive')
    return True
  except OSError:
    return False

try:
  drive.mount('/content/gdrive')
except OSError:
  pass                            # mount() itself can trip over a dead endpoint

if not drive_alive():
  # errno 107 (Transport endpoint is not connected): the mount is present but broken
  print('\033[1;33mDrive mount is stale, forcing a remount...')
  try:
    os.chdir('/content')          # never hold a working directory on the mount being replaced
    drive.mount('/content/gdrive', force_remount=True)
  except OSError:
    pass

if not drive_alive():
  # force_remount cannot repair an endpoint the kernel has already torn down
  raise SystemExit(
      '\033[1;31m\u2718 Google Drive could not be mounted, and a forced remount did not fix it.\n'
      '  Runtime > Restart runtime (or Disconnect and delete runtime), then run the cells from the top.\n'
      '  Nothing on Drive is lost - only the connection between this runtime and Drive.')

if Shared_Drive!="" and os.path.exists("/content/gdrive/Shareddrives"):
  mainpth="Shareddrives/"+Shared_Drive
else:
  mainpth="MyDrive"

clear_output()
inf('\u2714 Done','success', '50px')

#@markdown ---

Button(button_style='success', description='✔ Done', disabled=True, layout=Layout(min_width='50px'), style=But…

In [2]:
#@markdown # Install/Update AUTOMATIC1111 repo
from IPython.utils import capture
from IPython.display import clear_output
from subprocess import getoutput
import ipywidgets as widgets
import sys
import fileinput
import os
import time
import base64
import requests
from urllib.request import urlopen, Request
from urllib.parse import urlparse, parse_qs, unquote
from tqdm import tqdm
import six


blsaphemy=base64.b64decode(("ZWJ1aQ==").encode('ascii')).decode('ascii')

if not os.path.exists("/content/gdrive"):
  print('[1;31mGdrive not connected, using temporary colab storage ...')
  time.sleep(4)
  mainpth="MyDrive"
  !mkdir -p /content/gdrive/$mainpth
  Shared_Drive=""

if Shared_Drive!="" and not os.path.exists("/content/gdrive/Shareddrives"):
  print('[1;31mShared drive not detected, using default MyDrive')
  mainpth="MyDrive"

with capture.capture_output() as cap:
  def inf(msg, style, wdth): inf = widgets.Button(description=msg, disabled=True, button_style=style, layout=widgets.Layout(min_width=wdth));display(inf)
  fgitclone = "git clone --depth 1"
  !git clone -q --depth 1 --branch main https://github.com/TheLastBen/diffusers
  %mkdir -p /content/gdrive/$mainpth/sd
  %cd /content/gdrive/$mainpth/sd
  !git clone -q --branch master https://github.com/AUTOMATIC1111/stable-diffusion-w$blsaphemy
  !mkdir -p /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/cache/
  os.environ['TRANSFORMERS_CACHE']=f"/content/gdrive/{mainpth}/sd/stable-diffusion-w"+blsaphemy+"/cache"
  os.environ['TORCH_HOME'] = f"/content/gdrive/{mainpth}/sd/stable-diffusion-w"+blsaphemy+"/cache"
  !mkdir -p /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/repositories
  !git clone https://github.com/AUTOMATIC1111/stable-diffusion-w$blsaphemy-assets /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/repositories/stable-diffusion-webui-assets

with capture.capture_output() as cap:
  %cd /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/
  !git reset --hard
  !git checkout master
  time.sleep(1)
  !rm webui.sh
  !git pull
clear_output()
inf('\u2714 Done','success', '50px')

#@markdown ---

Button(button_style='success', description='✔ Done', disabled=True, layout=Layout(min_width='50px'), style=But…

In [ ]:
#@markdown # Requirements

import sys, sysconfig, os

# Colab's Python version moves over time (3.12 -> 3.13 as of 2026-08). Never hardcode it here:
# these paths are handed to rm/sed, which fail silently inside capture.capture_output(),
# so a stale version string disables every patch below without any error. See bugs/BUG-0015.md.
pyver   = f"python3.{sys.version_info.minor}"
sitepkg = sysconfig.get_paths()['purelib']
if not os.path.isdir(sitepkg):
  sitepkg = f"/usr/local/lib/{pyver}/dist-packages"
stdlib  = sysconfig.get_paths()['stdlib']

print('[1;32mInstalling requirements...')

with capture.capture_output() as cap:
  !rm -r $sitepkg/gradio*
  %cd /content/
  !wget -q -i https://raw.githubusercontent.com/TheLastBen/fast-stable-diffusion/main/Dependencies/A1111.txt
  !dpkg -i *.deb
  if not os.path.exists('/content/gdrive/'+mainpth+'/sd/stablediffusion'):
    !tar -C /content/gdrive/$mainpth --zstd -xf sd_mrep.tar.zst
  !tar -C / --zstd -xf gcolabdeps.tar.zst
  !rm *.deb | rm *.zst | rm *.txt
  if not os.path.exists('gdrive/'+mainpth+'/sd/libtcmalloc/libtcmalloc_minimal.so.4'):
    %env CXXFLAGS=-std=c++14
    !wget -q https://github.com/gperftools/gperftools/releases/download/gperftools-2.5/gperftools-2.5.tar.gz && tar zxf gperftools-2.5.tar.gz && mv gperftools-2.5 gperftools
    !wget -q https://github.com/TheLastBen/fast-stable-diffusion/raw/main/AUTOMATIC1111_files/Patch
    %cd /content/gperftools
    !patch -p1 < /content/Patch
    !./configure --enable-minimal --enable-libunwind --enable-frame-pointers --enable-dynamic-sized-delete-support --enable-sized-delete --enable-emergency-malloc; make -j4
    !mkdir -p /content/gdrive/$mainpth/sd/libtcmalloc && cp .libs/libtcmalloc*.so* /content/gdrive/$mainpth/sd/libtcmalloc
    %env LD_PRELOAD=/content/gdrive/$mainpth/sd/libtcmalloc/libtcmalloc_minimal.so.4
    %cd /content
    !rm *.tar.gz Patch && rm -r /content/gperftools
  else:
    %env LD_PRELOAD=/content/gdrive/$mainpth/sd/libtcmalloc/libtcmalloc_minimal.so.4

  !pip uninstall jax -y
  !pip install --force-reinstall sentencepiece -qq
  # gcolabdeps.tar.zst now bundles gradio 4.x, which removed gr.components.IOComponent; webui needs 3.41.2
  !pip install gradio==3.41.2 -qq
  # uvicorn 0.50+ always picks its sans-I/O websocket implementation, which creates one
  #   asyncio.Queue per connection on uvicorn's own event loop. gradio 3.41.2 runs its queue
  #   coroutine on a different loop, so touching that Queue raises
  #   '<Queue ...> is bound to a different event loop' - swallowed by a bare print(e) in
  #   gradio/queueing.py, so results silently never reach the browser.
  # 0.49.0 is the last release defaulting to the legacy implementation, which has no such Queue.
  # Verified by installing both against websockets 11.0.3 (the version gradio pins). BUG-0022.
  !pip install uvicorn==0.49.0 -qq
  # UNDER EVALUATION (BUG-0021): gradio 3.41.2's queue is built on anyio 3 primitives -
  #   run_coro_in_background, blocking portals, cross-loop handoff - all of which anyio 4
  #   reworked, and the queue has failed to deliver any result since the 2026-08-24 Colab
  #   image update. Every starlette in range, 0.50.0 included, declares anyio<5,>=3.6.2, so
  #   anyio can be moved on its own. pip will warn that google-genai and httpx2 want anyio 4;
  #   neither is on webui's path. Remove this pin if it turns out not to fix the queue.
  !pip install anyio==3.7.1 -qq
  # Packages removed from Colab base image (confirmed missing, see bugs/BUG-0013.md)
  !pip install pyngrok pytorch_lightning kornia diskcache gitpython piexif pillow-avif-plugin -qq
  # open_clip 3.x switched its transformer to batch_first; sgm's text_transformer_forward still
  # feeds LND tensors and a [77,77] attn_mask, so the newer layout raises
  #   RuntimeError: The shape of the 2D attn_mask is torch.Size([77, 77]), but should be (1, 1)
  # 2.20.0 is A1111's own pin and the last line that keeps the permute(1,0,2) layout. BUG-0020.
  !pip install open_clip_torch==2.20.0 -qq
  # Remaining packages from webui requirements.txt not guaranteed by Colab
  !pip install blendmodes clean-fid einops facexlib inflection jsonmerge lark omegaconf protobuf==3.20.0 psutil resize-right safetensors scikit-image tomesd torchdiffeq torchsde -qq
  # k-diffusion deps (clip-anytorch supersedes openai-clip; dctorch is k-diffusion specific)
  !pip install clip-anytorch dctorch -qq
  # sgm (generative-models) deps not in webui requirements.txt
  !pip install invisible-watermark natsort fairscale fire -qq
  # One pip install per line. Bundling these aborted the whole command on the first unresolvable
  # pin and silently took the other packages down with it (BUG-0015).
  # webui modules/api/models.py uses the pydantic v1 API (DynamicModel.__config__); Colab ships v2.
  # 1.10.22 is the first 1.10.x with cp313 wheels and the ForwardRef(recursive_guard) fix.
  !pip install pydantic==1.10.22 --no-deps -qq
  # fastapi must match pydantic. FastAPI dropped pydantic v1 in 0.126.0 (its types.py then does
  # `from pydantic.main import IncEx`, which only exists in v2) -> ImportError at gradio import.
  # 0.125.0 is the last release carrying the v1 branch (_compat/v1.py). BUG-0016.
  # UNDER EVALUATION (BUG-0021): lowered from 0.125.0 to 0.103.2 to bring **starlette** back to
  #   0.27.0. starlette is never installed directly - it arrives with fastapi - and 0.125.0
  #   drags 0.50.0, while gradio 3.41.2's queue was written against 0.26/0.27. Together with
  #   anyio==3.7.1 this is the ASGI layer gradio 3.41.2 actually shipped against. 0.103.2 still
  #   accepts pydantic v1, so BUG-0016 stays satisfied. Verified locally: 0.103.2 resolves to
  #   starlette 0.27.0 and the fastapi/starlette websocket API imports cleanly.
  !pip install fastapi==0.103.2 -qq
  # transformers 5.x removed CLIPTextModel.text_model, which webui's sd_hijack reaches into, and
  # reshaped _load_pretrained_model, which webui monkeypatches by position. 4.49.0 is the last
  # release whose 4th positional arg is still resolved_archive_file (4.50+ made it
  # pretrained_model_name_or_path). A1111 asks for 4.30.2, but that needs tokenizers 0.13.x,
  # which has no Python 3.13 wheel. Installed with deps: transformers 4.x needs huggingface_hub<1.0.
  # BUG-0019.
  !pip install transformers==4.49.0 -qq
  # sd-webui-controlnet annotator/mobile_sam imports controlnet_aux
  !pip install controlnet_aux --no-deps -qq
  !pip install wandb==0.15.12 --no-deps -qq
  !pip install scipy==1.15.3 --no-deps -qq
  # numpy is deliberately NOT pinned: webui asks for 1.26.x, which has no cp313 wheel and no
  # pure-python fallback, so pip falls back to building it from sdist and fails (BUG-0015).
  !pip install diffusers accelerate -U --no-deps -qq
  !rm -r $sitepkg/tensorflow*
  os.environ['PYTHONWARNINGS'] = 'ignore'
  !sed -i 's@text = _formatwarnmsg(msg)@text =\"\"@g' $stdlib/warnings.py
  !sed -i 's@from pytorch_lightning.loggers.wandb import WandbLogger  # noqa: F401@@g' $sitepkg/pytorch_lightning/loggers/__init__.py
  !sed -i 's@from .mailbox import ContextCancelledError@@g' $sitepkg/wandb/sdk/lib/retry.py
  !sed -i 's@raise ContextCancelledError("retry timeout")@print("retry timeout")@g' $sitepkg/wandb/sdk/lib/retry.py
  # no-op on pydantic 1.10.18+ (fixed upstream); kept in case the pin is ever lowered
  !sed -i 's@globalns, localns, set()@globalns, localns, recursive_guard=set()@g' $sitepkg/pydantic/typing.py
  !sed -i 's@position_embeddings = self.position_embedding(position_ids)@position_embeddings = self.position_embedding(position_ids.long())@g' $sitepkg/transformers/models/clip/modeling_clip.py
  # pytorch_lightning 2.x removed utilities.distributed; rank_zero_only moved to utilities.rank_zero
  !sed -i 's@from pytorch_lightning.utilities.distributed import rank_zero_only@from pytorch_lightning.utilities.rank_zero import rank_zero_only@g' /content/gdrive/$mainpth/sd/stablediffusion/ldm/models/diffusion/ddpm.py

clear_output()

# pip failures above are invisible (capture.capture_output + -qq), and a missing package does
# not surface until the WebUI dies minutes later. State the result here instead (BUG-0015).
import importlib.util
from importlib.metadata import version, PackageNotFoundError

checks = [('pydantic', 'pydantic', '1.10.'), ('gradio', 'gradio', '3.41.2'),
          ('fastapi', 'fastapi', '0.103.'), ('starlette', 'starlette', '0.27.'),
          ('transformers', 'transformers', '4.49.'),
          ('open_clip', 'open_clip_torch', '2.20.'),
          ('uvicorn', 'uvicorn', '0.49.'), ('websockets', 'websockets', '11.'),
          ('anyio', 'anyio', '3.7.'),
          ('controlnet_aux', 'controlnet_aux', None), ('pyngrok', 'pyngrok', None),
          ('pytorch_lightning', 'pytorch-lightning', None), ('kornia', 'kornia', None),
          ('clip', 'clip-anytorch', None),
          ('diskcache', 'diskcache', None), ('git', 'GitPython', None)]

problems = []
for mod, dist, want in checks:
  if importlib.util.find_spec(mod) is None:
    problems.append(f'{mod}: not installed')
    continue
  if want:
    try:
      got = version(dist)
    except PackageNotFoundError:
      got = '?'
    if not got.startswith(want):
      problems.append(f'{mod}: {got} installed, {want}x required')

# always report what is actually installed, not only what is missing - several debugging rounds
# have hinged on the question "did the Requirements cell even run?" (BUG-0022)
installed = []
for mod, dist, _ in checks:
  try:
    installed.append(f'{dist}=={version(dist)}')
  except PackageNotFoundError:
    installed.append(f'{dist}=MISSING')

print(f"\033[0;33mPython {sys.version.split()[0]}  |  site-packages: {sitepkg}")
print('\033[0;33m' + '  '.join(installed))
if problems:
  print('\033[1;31m\u2718 Requirements incomplete - the WebUI will not start:')
  for p in problems:
    print(f'\033[1;31m    - {p}')
else:
  inf('\u2714 Done','success', '50px')

#@markdown ---

Button(button_style='success', description='✔ Done', disabled=True, layout=Layout(min_width='50px'), style=But…

In [4]:
#@markdown # Fix xformers (PyTorch version mismatch)
import torch
from IPython.display import clear_output
import ipywidgets as widgets

def inf(msg, style, wdth): inf = widgets.Button(description=msg, disabled=True, button_style=style, layout=widgets.Layout(min_width=wdth));display(inf)

print(f"\033[0;33mInstalled PyTorch: {torch.__version__}")
print("\033[0;33mReinstalling xformers for current PyTorch...")
!pip install xformers --upgrade -q
clear_output()
inf('\u2714 xformers reinstalled','success', '250px')

#@markdown ---

Button(button_style='success', description='✔ xformers reinstalled', disabled=True, layout=Layout(min_width='2…

In [6]:
#@markdown # Model Download/Load

import gdown
from gdown.download import get_url_from_gdrive_confirmation
import re

Use_Temp_Storage = False #@param {type:"boolean"}
#@markdown - If not, make sure you have enough space on your gdrive

#@markdown ---

Model_Version = "SDXL" #@param ["SDXL", "1.5", "v1.5 Inpainting", "V2.1-768px"]

#@markdown Or
PATH_to_MODEL = "" #@param {type:"string"}
#@markdown - Insert the full path of your custom model or to a folder containing multiple models

#@markdown Or
MODEL_LINK = "" #@param {type:"string"}


def getsrc(url):
    parsed_url = urlparse(url)
    if parsed_url.netloc == 'civitai.com':
        src='civitai'
    elif parsed_url.netloc == 'drive.google.com':
        src='gdrive'
    elif parsed_url.netloc == 'huggingface.co':
        src='huggingface'
    else:
        src='others'
    return src

src=getsrc(MODEL_LINK)

def get_name(url, gdrive):
    if not gdrive:
        response = requests.get(url, allow_redirects=False)
        if "Location" in response.headers:
            redirected_url = response.headers["Location"]
            quer = parse_qs(urlparse(redirected_url).query)
            if "response-content-disposition" in quer:
                disp_val = quer["response-content-disposition"][0].split(";")
                for vals in disp_val:
                    if vals.strip().startswith("filename="):
                        filenm=unquote(vals.split("=", 1)[1].strip())
                        return filenm.replace("\"","")
    else:
        headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36"}
        lnk="https://drive.google.com/uc?id={id}&export=download".format(id=url[url.find("/d/")+3:url.find("/view")])
        res = requests.session().get(lnk, headers=headers, stream=True, verify=True)
        res = requests.session().get(get_url_from_gdrive_confirmation(res.text), headers=headers, stream=True, verify=True)
        content_disposition = six.moves.urllib_parse.unquote(res.headers["Content-Disposition"])
        filenm = re.search('attachment; filename="(.*?)"', content_disposition).groups()[0]
        return filenm


def dwn(url, dst, msg):
    file_size = None
    req = Request(url, headers={"User-Agent": "torch.hub"})
    u = urlopen(req)
    meta = u.info()
    if hasattr(meta, 'getheaders'):
        content_length = meta.getheaders("Content-Length")
    else:
        content_length = meta.get_all("Content-Length")
    if content_length is not None and len(content_length) > 0:
        file_size = int(content_length[0])

    with tqdm(total=file_size, disable=False, mininterval=0.5,
              bar_format=msg+' |{bar:20}| {percentage:3.0f}%') as pbar:
        with open(dst, "wb") as f:
            while True:
                buffer = u.read(8192)
                if len(buffer) == 0:
                    break
                f.write(buffer)
                pbar.update(len(buffer))
            f.close()


def sdmdls(ver, Use_Temp_Storage):

  if ver=='1.5':
    if Use_Temp_Storage:
      os.makedirs('/content/temp_models', exist_ok=True)
      model='/content/temp_models/v1-5-pruned-emaonly.safetensors'
    else:
      model='/content/gdrive/'+mainpth+'/sd/stable-diffusion-w'+blsaphemy+'/models/Stable-diffusion/v1-5-pruned-emaonly.safetensors'
    link='https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors'
  elif ver=='V2.1-768px':
    if Use_Temp_Storage:
      os.makedirs('/content/temp_models', exist_ok=True)
      model='/content/temp_models/v2-1_768-ema-pruned.safetensors'
    else:
      model='/content/gdrive/'+mainpth+'/sd/stable-diffusion-w'+blsaphemy+'/models/Stable-diffusion/v2-1_768-ema-pruned.safetensors'
    link='https://huggingface.co/stabilityai/stable-diffusion-2-1/resolve/main/v2-1_768-ema-pruned.safetensors'
  elif ver=='v1.5 Inpainting':
    if Use_Temp_Storage:
      os.makedirs('/content/temp_models', exist_ok=True)
      model='/content/temp_models/sd-v1-5-inpainting.ckpt'
    else:
      model='/content/gdrive/'+mainpth+'/sd/stable-diffusion-w'+blsaphemy+'/models/Stable-diffusion/sd-v1-5-inpainting.ckpt'
    link='https://huggingface.co/runwayml/stable-diffusion-inpainting/resolve/main/sd-v1-5-inpainting.ckpt'
  elif ver=='SDXL':
    if Use_Temp_Storage:
      os.makedirs('/content/temp_models', exist_ok=True)
      model='/content/temp_models/sd_xl_base_1.0.safetensors'
    else:
      model='/content/gdrive/'+mainpth+'/sd/stable-diffusion-w'+blsaphemy+'/models/Stable-diffusion/sd_xl_base_1.0.safetensors'
    link='https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors'

  if not os.path.exists(model):
    !gdown --fuzzy -O $model $link
    if os.path.exists(model):
      clear_output()
      inf('\u2714 Done','success', '50px')
    else:
      inf('\u2718 Something went wrong, try again','danger', "250px")
  else:
      clear_output()
      inf('\u2714 Model already exists','primary', '300px')

  return model


if (PATH_to_MODEL !=''):
  if os.path.exists(str(PATH_to_MODEL)):
    inf('\u2714 Using the trained model.','success', '200px')

  else:
      while not os.path.exists(str(PATH_to_MODEL)):
        inf('\u2718 Wrong path, use the colab file explorer to copy the path : ','danger', "400px")
        PATH_to_MODEL=input()
      if os.path.exists(str(PATH_to_MODEL)):
        inf('\u2714 Using the custom model.','success', '200px')

  model=PATH_to_MODEL

elif MODEL_LINK != "":

      if src=='civitai':
         modelname=get_name(MODEL_LINK, False)
         if Use_Temp_Storage:
            os.makedirs('/content/temp_models', exist_ok=True)
            model=f'/content/temp_models/{modelname}'
         else:
            model=f'/content/gdrive/{mainpth}/sd/stable-diffusion-w{blsaphemy}/models/Stable-diffusion/{modelname}'
         if not os.path.exists(model):
            dwn(MODEL_LINK, model, 'Downloading the custom model')
            clear_output()
         else:
            inf('\u2714 Model already exists','primary', '300px')
      elif src=='gdrive':
         modelname=get_name(MODEL_LINK, True)
         if Use_Temp_Storage:
            os.makedirs('/content/temp_models', exist_ok=True)
            model=f'/content/temp_models/{modelname}'
         else:
            model=f'/content/gdrive/{mainpth}/sd/stable-diffusion-w{blsaphemy}/models/Stable-diffusion/{modelname}'
         if not os.path.exists(model):
            gdown.download(url=MODEL_LINK, output=model, quiet=False, fuzzy=True)
            clear_output()
         else:
            inf('\u2714 Model already exists','primary', '300px')
      else:
         modelname=os.path.basename(MODEL_LINK)
         if Use_Temp_Storage:
            os.makedirs('/content/temp_models', exist_ok=True)
            model=f'/content/temp_models/{modelname}'
         else:
            model=f'/content/gdrive/{mainpth}/sd/stable-diffusion-w{blsaphemy}/models/Stable-diffusion/{modelname}'
         if not os.path.exists(model):
            gdown.download(url=MODEL_LINK, output=model, quiet=False, fuzzy=True)
            clear_output()
         else:
            inf('\u2714 Model already exists','primary', '700px')

      if os.path.exists(model) and os.path.getsize(model) > 1810671599:
        inf('\u2714 Model downloaded, using the custom model.','success', '300px')
      else:
        !rm model
        inf('\u2718 Wrong link, check that the link is valid','danger', "300px")

else:
  model=sdmdls(Model_Version, Use_Temp_Storage)

#@markdown ---

Button(button_style='primary', description='✔ Model already exists', disabled=True, layout=Layout(min_width='3…

In [7]:
#@markdown # Download LoRA

LoRA_LINK = "" #@param {type:"string"}

if LoRA_LINK == "":
  inf('\u2714 Nothing to do','primary', '200px')
else:
  os.makedirs('/content/gdrive/'+mainpth+'/sd/stable-diffusion-w'+blsaphemy+'/models/Lora', exist_ok=True)

  src=getsrc(LoRA_LINK)

  if src=='civitai':
      modelname=get_name(LoRA_LINK, False)
      loramodel=f'/content/gdrive/{mainpth}/sd/stable-diffusion-w{blsaphemy}/models/Lora/{modelname}'
      if not os.path.exists(loramodel):
        dwn(LoRA_LINK, loramodel, 'Downloading the LoRA model '+modelname)
        clear_output()
      else:
        inf('\u2714 Model already exists','primary', '200px')
  elif src=='gdrive':
      modelname=get_name(LoRA_LINK, True)
      loramodel=f'/content/gdrive/{mainpth}/sd/stable-diffusion-w{blsaphemy}/models/Lora/{modelname}'
      if not os.path.exists(loramodel):
        gdown.download(url=LoRA_LINK, output=loramodel, quiet=False, fuzzy=True)
        clear_output()
      else:
        inf('\u2714 Model already exists','primary', '200px')
  else:
      modelname=os.path.basename(LoRA_LINK)
      loramodel=f'/content/gdrive/{mainpth}/sd/stable-diffusion-w{blsaphemy}/models/Lora/{modelname}'
      if not os.path.exists(loramodel):
        gdown.download(url=LoRA_LINK, output=loramodel, quiet=False, fuzzy=True)
        clear_output()
      else:
        inf('\u2714 Model already exists','primary', '200px')

  if os.path.exists(loramodel) :
    inf('\u2714 LoRA downloaded','success', '200px')
  else:
    inf('\u2718 Wrong link, check that the link is valid','danger', "300px")

#@markdown ---

Button(button_style='primary', description='✔ Nothing to do', disabled=True, layout=Layout(min_width='200px'),…

In [8]:
#@markdown # ControlNet
from torch.hub import download_url_to_file
from urllib.parse import urlparse
import re
from subprocess import run

XL_Model = "None" #@param [ "None", "All", "Canny", "Depth", "Sketch", "OpenPose", "Recolor"]

v1_Model = "None" #@param [ "None", "All (21GB)", "Canny", "Depth", "Lineart", "MLSD", "Normal", "OpenPose", "Scribble", "Seg", "ip2p", "Shuffle", "Inpaint", "Softedge", "Lineart_Anime", "Tile", "T2iadapter_Models"]

v2_Model = "None" #@param [ "None", "All", "Canny", "Depth", "HED", "OpenPose", "Scribble"]

#@markdown - Download/update ControlNet extension and its models

def download(url, model_dir):

    filename = os.path.basename(urlparse(url).path)
    pth = os.path.abspath(os.path.join(model_dir, filename))
    if not os.path.exists(pth):
        print('Downloading: '+os.path.basename(url))
        download_url_to_file(url, pth, hash_prefix=None, progress=True)
    else:
      print(f"[1;32mThe model {filename} already exists[0m")


Canny='https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/diffusers_xl_canny_mid.safetensors'
Depth='https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/diffusers_xl_depth_mid.safetensors'
Sketch='https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/sai_xl_sketch_256lora.safetensors'
OpenPose='https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/thibaud_xl_openpose_256lora.safetensors'
Recolor='https://huggingface.co/lllyasviel/sd_control_collection/resolve/main/sai_xl_recolor_128lora.safetensors'


with capture.capture_output() as cap:
  %cd /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/extensions
  if not os.path.exists('sd-w'+blsaphemy+'-controlnet'):
    !git clone https://github.com/Mikubill/sd-w$blsaphemy-controlnet.git
    %cd /content
  else:
    %cd sd-w$blsaphemy-controlnet
    !git reset --hard
    !git pull
    %cd /content

mdldir='/content/gdrive/'+mainpth+'/sd/stable-diffusion-w'+blsaphemy+'/extensions/sd-w'+blsaphemy+'-controlnet/models'
for filename in os.listdir(mdldir):
  if "_sd14v1" in filename:
    renamed = re.sub("_sd14v1", "-fp16", filename)
    os.rename(os.path.join(mdldir, filename), os.path.join(mdldir, renamed))

!wget -q -O CN_models.txt https://github.com/TheLastBen/fast-stable-diffusion/raw/main/AUTOMATIC1111_files/CN_models.txt
!wget -q -O CN_models_v2.txt https://github.com/TheLastBen/fast-stable-diffusion/raw/main/AUTOMATIC1111_files/CN_models_v2.txt
!wget -q -O CN_models_XL.txt https://github.com/TheLastBen/fast-stable-diffusion/raw/main/AUTOMATIC1111_files/CN_models_XL.txt


with open("CN_models.txt", 'r') as f:
  mdllnk = f.read().splitlines()
with open("CN_models_v2.txt", 'r') as d:
  mdllnk_v2 = d.read().splitlines()
with open("CN_models_XL.txt", 'r') as d:
  mdllnk_XL = d.read().splitlines()

!rm CN_models.txt CN_models_v2.txt CN_models_XL.txt


if XL_Model == "All":
  for lnk_XL in mdllnk_XL:
      download(lnk_XL, mdldir)
  clear_output()
  inf('\u2714 Done','success', '50px')

elif XL_Model == "None":
    pass
    clear_output()
    inf('\u2714 Done','success', '50px')

else:
  download(globals()[XL_Model], mdldir)
  clear_output()
  inf('\u2714 Done','success', '50px')


Canny='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_canny.pth'
Depth='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11f1p_sd15_depth.pth'
Lineart='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_lineart.pth'
MLSD='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_mlsd.pth'
Normal='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_normalbae.pth'
OpenPose='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_openpose.pth'
Scribble='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_scribble.pth'
Seg='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_seg.pth'
ip2p='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11e_sd15_ip2p.pth'
Shuffle='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11e_sd15_shuffle.pth'
Inpaint='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_inpaint.pth'
Softedge='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_softedge.pth'
Lineart_Anime='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15s2_lineart_anime.pth'
Tile='https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11f1e_sd15_tile.pth'


with capture.capture_output() as cap:
  cfgnames=[os.path.basename(url).split('.')[0]+'.yaml' for url in mdllnk_v2]
  %cd /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/extensions/sd-w$blsaphemy-controlnet/models
  for name in cfgnames:
      run(['cp', 'cldm_v21.yaml', name])
  %cd /content

if v1_Model == "All (21GB)":
  for lnk in mdllnk:
      download(lnk, mdldir)
  clear_output()

elif v1_Model == "T2iadapter_Models":
  mdllnk=list(filter(lambda x: 't2i' in x, mdllnk))
  for lnk in mdllnk:
      download(lnk, mdldir)
  clear_output()

elif v1_Model == "None":
    pass
    clear_output()

else:
  download(globals()[v1_Model], mdldir)
  clear_output()

Canny='https://huggingface.co/thibaud/controlnet-sd21/resolve/main/control_v11p_sd21_canny.safetensors'
Depth='https://huggingface.co/thibaud/controlnet-sd21/resolve/main/control_v11p_sd21_depth.safetensors'
HED='https://huggingface.co/thibaud/controlnet-sd21/resolve/main/control_v11p_sd21_hed.safetensors'
OpenPose='https://huggingface.co/thibaud/controlnet-sd21/resolve/main/control_v11p_sd21_openposev2.safetensors'
Scribble='https://huggingface.co/thibaud/controlnet-sd21/resolve/main/control_v11p_sd21_scribble.safetensors'


if v2_Model == "All":
  for lnk_v2 in mdllnk_v2:
      download(lnk_v2, mdldir)
  clear_output()
  inf('\u2714 Done','success', '50px')

elif v2_Model == "None":
    pass
    clear_output()
    inf('\u2714 Done','success', '50px')

else:
  download(globals()[v2_Model], mdldir)
  clear_output()
  inf('\u2714 Done','success', '50px')

  #@markdown ---

Button(button_style='success', description='✔ Done', disabled=True, layout=Layout(min_width='50px'), style=But…

In [9]:
#@markdown # Start Stable-Diffusion
from IPython.utils import capture
import time
import sys
import sysconfig
import fileinput
from pyngrok import ngrok, conf
import re

# same reason as the Requirements cell: Colab's Python version is not ours to hardcode (BUG-0015)
sitepkg = sysconfig.get_paths()['purelib']
if not os.path.isdir(sitepkg):
  sitepkg = f"/usr/local/lib/python3.{sys.version_info.minor}/dist-packages"


Ngrok_token = "" #@param {type:"string"}

#@markdown - Input your ngrok token if you want to use ngrok server

User = "" #@param {type:"string"}
Password= "" #@param {type:"string"}
#@markdown - Add credentials to your Gradio interface (optional)

#@markdown - Disable_Gradio_Queue is ON by default because gradio's websocket queue does not
#@markdown   deliver results in this environment: with it off, generation finishes but the
#@markdown   gallery stays empty, PNG Info stops working and Save fails (BUG-0021). With it on,
#@markdown   results come back over plain HTTP and all of that works.
#@markdown - The trade-off: each generation becomes one long synchronous POST, and the
#@markdown   gradio.live tunnel cuts those off with a 504 once a run takes a few minutes. For
#@markdown   long multi-batch runs, use your own tunnel by filling in Ngrok_token above.
Disable_Gradio_Queue = True #@param {type:"boolean"}
queue_arg = "--no-gradio-queue" if Disable_Gradio_Queue else ""

#@markdown - Test_Disable_Extensions launches without any third-party extension. Use it to find
#@markdown   out whether an extension is what stops results from reaching the page (BUG-0021)
Test_Disable_Extensions = False #@param {type:"boolean"}
ext_arg = "--disable-extra-extensions" if Test_Disable_Extensions else ""

#@markdown - Test_Stock_Gradio keeps gradio's own blocks.py instead of the patched copy this
#@markdown   notebook normally installs. That patch rewrites launch(), including how the
#@markdown   protocol and server name are decided - the last modification to gradio itself
#@markdown   that has never been tested without (BUG-0021). Note it also removes the
#@markdown   "Connected" banner and is required by the Ngrok path, so leave it off normally.
Test_Stock_Gradio = False #@param {type:"boolean"}

auth=f"--gradio-auth {User}:{Password}"
if User =="" or Password=="":
  auth=""


# Google Drive is a FUSE mount and it can drop mid-session. Once it does, everything downstream
# fails with a cryptic "OSError: [Errno 107] Transport endpoint is not connected" - raised by
# something as innocent as os.getcwd() inside `import gradio`. Check it here instead. BUG-0018.
try:
  os.getcwd()
except OSError:
  os.chdir('/content')            # the working directory itself was on the dead mount
sdpath = f'/content/gdrive/{mainpth}/sd/stable-diffusion-w{blsaphemy}'

# BUG-0021 instrument: gradio/queueing.py decides here whether a finished job reaches the browser,
#   async def send_message(self, event, data, timeout=1):
#       try: await asyncio.wait_for(event.websocket.send_json(data), timeout=timeout)
#       except Exception: await self.clean_event(event); return False
# and it swallows the exception without a word - no print, no log, and the caller ignores the
# return value. An external probe shows the server delivering fine, so make the browser's own
# events say whether their result actually went out.
_qp = sitepkg + '/gradio/queueing.py'
_patches = [
  # BUG-0023, the actual fix. gradio's queue does not run the prediction in-process: it POSTs to
  # its own /api/predict and waits for the answer. That client is built with httpx's DEFAULT
  # timeout of 5 seconds, so any prediction longer than five seconds - every image generation -
  # raises ReadTimeout. AsyncRequest catches it silently ("Exceptions are handled silently" is in
  # its own docstring), the queue reports process_completed with success=False, and the browser
  # correctly refuses to apply a failed result. Generation finishes and the PNG is written, but
  # the page never sees it.
  ('self.queue_client = httpx.AsyncClient(verify=ssl_verify)',
   'self.queue_client = httpx.AsyncClient(verify=ssl_verify, timeout=None)',
   'queue client timeout removed (BUG-0023)'),
  # make the failure audible if it ever happens again
  ("            if response.has_exception:\n",
   "            if response.has_exception:\n"
   "                print(f'[gradio] call_prediction FAILED: "
   "{type(response.exception).__name__}: {response.exception}', flush=True)\n",
   'call_prediction failures now logged'),
  # and the send path, which swallows everything the same way
  ("        except Exception:\n"
   "            await self.clean_event(event)\n"
   "            return False",
   "        except Exception as e:\n"
   "            print(f'[gradio] send_message FAILED: {type(e).__name__}: {e}', flush=True)\n"
   "            await self.clean_event(event)\n"
   "            return False",
   'send_message failures now logged'),
]
try:
  _src = open(_qp, encoding='utf-8').read()
  _done = []
  for _old, _new, _label in _patches:
    if _new in _src:
      _done.append(_label + ' [already]')
    elif _old in _src:
      _src = _src.replace(_old, _new, 1)
      _done.append(_label)
    else:
      _done.append('\033[1;31mNOT FOUND: ' + _label)
  open(_qp, 'w', encoding='utf-8').write(_src)
  for _d in _done:
    print('\033[0;33m  gradio patch: ' + _d)
except Exception as _e:
  print(f'\033[1;31mCould not patch gradio queueing.py: {_e}')

if not os.path.isdir(sdpath):
  raise SystemExit(
      '\033[1;31m\u2718 Google Drive is not mounted, or the mount has dropped.\n'
      '  1. Re-run the "Connect Google Drive" cell (it force-remounts a stale mount).\n'
      '  2. If that does not help: Runtime > Restart runtime, then run the cells from the top.\n'
      f'  (looked for: {sdpath})')

with capture.capture_output() as cap:
  %cd /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/modules/
  !wget -q -O extras.py https://raw.githubusercontent.com/AUTOMATIC1111/stable-diffusion-w$blsaphemy/master/modules/extras.py
  !wget -q -O sd_models.py https://raw.githubusercontent.com/AUTOMATIC1111/stable-diffusion-w$blsaphemy/master/modules/sd_models.py
  if not Test_Stock_Gradio:
    !wget -q -O $sitepkg/gradio/blocks.py https://raw.githubusercontent.com/TheLastBen/fast-stable-diffusion/main/AUTOMATIC1111_files/blocks.py
  else:
    # restore whatever pip shipped, in case an earlier run in this runtime overwrote it
    !pip install --force-reinstall --no-deps gradio==3.41.2 -qq
  %cd /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/
  
  !sed -i 's@shared.opts.data\["sd_model_checkpoint"] = checkpoint_info.title@shared.opts.data\["sd_model_checkpoint"] = checkpoint_info.title;model.half()@' /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/modules/sd_models.py
  #!sed -i 's@ui.create_ui().*@ui.create_ui();shared.demo.queue(concurrency_count=999999,status_update_rate=0.1)@' /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/webui.py
  !sed -i "s@map_location='cpu'@map_location='cuda'@" /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/modules/extras.py

  !sed -i 's@possible_sd_paths =.*@possible_sd_paths = [\"/content/gdrive/{mainpth}/sd/stablediffusion\"]@' /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/modules/paths.py
  !sed -i 's@\.\.\/@src/@g' /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/modules/paths.py
  !sed -i 's@src/generative-models@generative-models@g' /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/modules/paths.py

  !sed -i 's@print(\"No module.*@@' /content/gdrive/$mainpth/sd/stablediffusion/ldm/modules/diffusionmodules/model.py
  !sed -i 's@\["sd_model_checkpoint"\]@\["sd_model_checkpoint", "sd_vae", "CLIP_stop_at_last_layers", "inpainting_mask_weight", "initial_noise_multiplier"\]@g' /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/modules/shared.py
  # TheLastBen's sed, restored (BUG-0020; BUG-0017 had reversed it). webui calls
  #   CLIPTextModel_from_pretrained(None, config=<name>, state_dict={})
  # but transformers 4.49 resolves the config from the *model name*, not from `config`, so a
  # None name asks the Hub for https://huggingface.co/None/... -> 401 -> the fast path dies and
  # webui re-downloads CLIP + OpenCLIP (~12 GB) every session. Passing the real name fixes that;
  # transformers 4.x has no objection to name + state_dict={} (that check is a 5.x addition,
  # which is why this sed had to be reversed while Colab was on 5.x - see BUG-0017/BUG-0019).
  # This is coupled to the transformers==4.49.0 pin: revisit both together.
  !sed -i "s@res = self.CLIPTextModel_from_pretrained(None@res = self.CLIPTextModel_from_pretrained(pretrained_model_name_or_path@" /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/modules/sd_disable_initialization.py

share=''
if Ngrok_token!="":
  ngrok.kill()
  srv=ngrok.connect(7860, pyngrok_config=conf.PyngrokConfig(auth_token=Ngrok_token) , bind_tls=True).public_url
  # the URL is otherwise only used to patch gradio's blocks.py, leaving nothing on screen
  # that says where to connect. It changes on every restart, so print it.
  print(f'[0;32mNgrok URL: {srv}')

  for line in fileinput.input(sitepkg + '/gradio/blocks.py', inplace=True):
    if line.strip().startswith('self.server_name ='):
        line = f'            self.server_name = "{srv[8:]}"\n'
    if line.strip().startswith('self.protocol = "https"'):
        line = '            self.protocol = "https"\n'
    if line.strip().startswith('if self.local_url.startswith("https") or self.is_colab'):
        line = ''
    if line.strip().startswith('else "http"'):
        line = ''
    sys.stdout.write(line)
else:
  share='--share'

ckptdir=''
if os.path.exists('/content/temp_models'):
  ckptdir='--ckpt-dir /content/temp_models'

# say which flags are actually in effect - a notebook parameter that silently fails to reach the
# launch line has cost us a whole A/B round before (BUG-0021)
print(f'[0;33mLaunch flags: {share} {queue_arg} {ext_arg}')
if Test_Stock_Gradio:
  print("[0;33mUsing stock gradio blocks.py (TheLastBen patch skipped)")

# and which versions are actually serving. The Requirements cell reports this too, but re-running
# it costs ten minutes, so a launch log that stands on its own is worth four lines (BUG-0022).
from importlib.metadata import version as _v, PackageNotFoundError as _NF
def _ver(d):
  try:
    return f'{d}=={_v(d)}'
  except _NF:
    return f'{d}=MISSING'
print('[0;33mServing stack: ' + '  '.join(_ver(d) for d in
      ['gradio', 'uvicorn', 'websockets', 'fastapi', 'starlette', 'pydantic', 'anyio']))

# BUG-0021: with gradio's queue enabled, generations complete but no result ever reaches the
# page - blank gallery, dead PNG Info, Save failing. Say so at launch rather than letting a
# run be spent rediscovering it.
if not Disable_Gradio_Queue:
  print('[1;31m\u26a0 gradio queue is ENABLED - results will not reach the page (BUG-0021).')
  print('[1;31m  Tick Disable_Gradio_Queue above unless you are deliberately testing the queue.')

try:
  model
  if os.path.isfile(model):
    !python /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/webui.py $share --api --disable-safe-unpickle --enable-insecure-extension-access --no-download-sd-model --no-half-vae  --ckpt "$model" --xformers $auth --disable-console-progressbars --skip-version-check $queue_arg $ext_arg $ckptdir
  else:
    !python /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/webui.py $share --api --disable-safe-unpickle --enable-insecure-extension-access --no-download-sd-model --no-half-vae  --ckpt-dir "$model" --xformers $auth --disable-console-progressbars --skip-version-check $queue_arg $ext_arg
except:
   !python /content/gdrive/$mainpth/sd/stable-diffusion-w$blsaphemy/webui.py $share --api --disable-safe-unpickle --enable-insecure-extension-access --no-download-sd-model --no-half-vae --xformers $auth --disable-console-progressbars --skip-version-check $queue_arg $ext_arg $ckptdir

WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.10.0+cu128 with CUDA 1208 (you have 2.11.0+cpu)
    Python  3.10.19 (you have 3.12.13)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details
Civitai Helper: Root Path is: /content/gdrive/MyDrive/sd/stable-diffusion-webui
Civitai Helper: Get Custom Model Folder
Installing None
Tag Autocomplete: Could not locate model-keyword extension, Lora trigger word completion will be limited to those added through the extra networks menu.
person_yolov8s-seg.pt  0% 0.00/23.9M [00:00<?, ?B/s]
yolov8x-worldv2.pt  0% 0.00/146M [00:00<?, ?B/s]

hand_yolov8n.pt  0% 0.00/6.24M [00:00<?, ?B/s]


person_yolov8n-seg.pt  0% 0.00/6.78M [00:00<?, ?B/s]



face_yolov8n.pt  0% 0.00/6.23M [00:00<?, ?B/s]




face_yolov8s.pt  0% 0.00/22.5M [00:00<?, ?B/s]
